# Actividad integradora — Clasificación de defectos (capsule, MVTec-AD)

**Consigna (actividad.md):** desarrollar una aplicación de visión computacional entrenando un modelo de clasificación con un dataset a elección, generando proceso de entrenamiento y métricas de performance, e informe final (problema, dataset, resultados, conclusiones).

Este notebook es el punto de partida: replica la estructura del ejemplo del profesor (`transfer_learning.ipynb`) pero sobre MVTec-AD `capsule` en clasificación **binaria** (buena vs. defectuosa) con una CNN sencilla (ver `informe.md`).

> Requiere: `pip install tensorflow scikit-learn matplotlib pillow`

## 1. Problema y dataset

**Problema (informe.md):** clasificar imágenes de productos industriales según su estado visual.

**Dataset:** MVTec-AD, categoría `capsule` — ver [proyecto original](https://www.mvtec.com/research-teaching/datasets/mvtec-ad) y `README.md` para la descarga. Estructura local en `datasets/mvtec-ad/capsule/`:

- `train/good/` — solo piezas buenas (el dataset original es no supervisado).
- `test/{crack, faulty_imprint, good, poke, scratch, squeeze}/` — buenas y defectuosas.
- `ground_truth/` — máscaras (no se usan en clasificación).

Para clasificación supervisada binaria se reagrupa todo en `good (0)` vs `defectuosa (1)` con split estratificado.

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import tensorflow as tf
from tensorflow.keras import Sequential, layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

In [ ]:
DATA_DIR = pathlib.Path("datasets/mvtec-ad/capsule")

# Conteo por carpeta (exploración rápida)
for split in ["train", "test"]:
    for cls in sorted((DATA_DIR / split).iterdir()):
        n = len(list(cls.glob("*.png")))
        print(f"{split}/{cls.name}: {n}")

# Muestra visual: buena vs. defectuosa
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (name, path) in zip(axes, [
        ("good", next((DATA_DIR / "test" / "good").glob("*.png"))),
        ("defectuosa (crack)", next((DATA_DIR / "test" / "crack").glob("*.png"))),
]):
    ax.imshow(plt.imread(path))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Carga y preprocessing

Igual que el ejemplo (`resize` a 150×150), pero desde disco: etiqueta `0` = `good`, `1` = cualquier defecto. Split estratificado 80/20 (el `fit` reserva además 20% de validación, como en el ejemplo).

In [ ]:
IMG_SIZE = (150, 150)

paths, labels = [], []
for split in ["train", "test"]:
    for cls_dir in sorted((DATA_DIR / split).iterdir()):
        y = 0 if cls_dir.name == "good" else 1
        for p in sorted(cls_dir.glob("*.png")):
            paths.append(p)
            labels.append(y)

X = np.stack([
    tf.keras.utils.img_to_array(tf.keras.utils.load_img(p, target_size=IMG_SIZE))
    for p in paths
]).astype("float32")  # el Rescaling del modelo la lleva a [0, 1]
y = np.array(labels)

print("X:", X.shape, "| buenas:", (y == 0).sum(), "| defectuosas:", (y == 1).sum())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("train:", X_train.shape, "test:", X_test.shape)

## 3. Modelo — CNN sencilla

Espejo del "Hand Made Model" del ejemplo (3 bloques Conv+Pool + Dense 50/20), con salida binaria `Dense(1, sigmoid)`.

In [ ]:
model = Sequential([
    layers.Rescaling(1. / 255, input_shape=(150, 150, 3)),
    layers.Conv2D(16, kernel_size=10, activation="relu"),
    layers.MaxPooling2D(3),
    layers.Conv2D(32, kernel_size=8, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(32, kernel_size=6, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(50, activation="relu"),
    layers.Dense(20, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.summary()

## 4. Entrenamiento

Mismo esquema del ejemplo: `adam` + `EarlyStopping(val_accuracy, patience=5)` hasta 50 epochs.

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

es = EarlyStopping(monitor="val_accuracy", mode="max", patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train, epochs=50, batch_size=32,
    validation_split=0.2, callbacks=[es],
)

In [ ]:
acc, val_acc = history.history["accuracy"], history.history["val_accuracy"]
loss, val_loss = history.history["loss"], history.history["val_loss"]
epochs = range(len(acc))

plt.plot(epochs, acc, "bo", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.figure()
plt.plot(epochs, loss, "bo", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

## 5. Evaluación y métricas

La consigna pide métricas de performance: además de `accuracy/loss` se agregan matriz de confusión y reporte (precisión/recall/F1), claves en inspección industrial donde un falso negativo cuesta más.

In [ ]:
print(model.evaluate(X_test, y_test, verbose=0))

y_pred = (model.predict(X_test, verbose=0).ravel() > 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=["good", "defectuosa"]))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred), display_labels=["good", "defectuosa"]
).plot()
plt.title("Matriz de confusión (test)")
plt.show()

## 6. Transfer Learning — VGG16 congelado

El mismo esquema del ejemplo del profesor (`transfer_learning.ipynb`, celdas 9–15): una base pre-entrenada en ImageNet aporta los filtros visuales y **solo el cabezal denso aprende** a distinguir buena vs. defectuosa.

Claves:
- `include_top=False` y `trainable = False` **antes** del `compile`.
- Se usa `preprocess_input` de VGG16 (paso a BGR + resta de media de ImageNet), **no** `Rescaling`.
- Los arrays `X_*` están en 0–255 (el `Rescaling` vive dentro de la CNN baseline), así que sirven directo.
- La primera ejecución descarga los pesos `imagenet` (~528 MB).


In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input

base_tl = VGG16(weights="imagenet", include_top=False, input_shape=(150, 150, 3))
base_tl.trainable = False  # se congela ANTES de compilar

model_tl = Sequential([
    base_tl,
    layers.Flatten(),  # igual que el ejemplo del profesor
    layers.Dense(50, activation="relu"),
    layers.Dense(20, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model_tl.summary()  # fijate en "non-trainable params": la base no aprende


In [ ]:
X_train_tl = preprocess_input(X_train.copy())
X_test_tl = preprocess_input(X_test.copy())

model_tl.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

es_tl = EarlyStopping(monitor="val_accuracy", mode="max", patience=5, restore_best_weights=True)

history_tl = model_tl.fit(
    X_train_tl, y_train, epochs=50, batch_size=32,
    validation_split=0.2, callbacks=[es_tl],
)


In [ ]:
acc, val_acc = history_tl.history["accuracy"], history_tl.history["val_accuracy"]
loss, val_loss = history_tl.history["loss"], history_tl.history["val_loss"]
epochs = range(len(acc))

plt.plot(epochs, acc, "bo", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy (TL)")
plt.legend()
plt.figure()
plt.plot(epochs, loss, "bo", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss (TL)")
plt.legend()
plt.show()


In [ ]:
print(model_tl.evaluate(X_test_tl, y_test, verbose=0))

y_pred_tl = (model_tl.predict(X_test_tl, verbose=0).ravel() > 0.5).astype(int)
print(classification_report(y_test, y_pred_tl, target_names=["good", "defectuosa"]))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_tl), display_labels=["good", "defectuosa"]
).plot()
plt.title("Matriz de confusión TL (test)")
plt.show()


## 7. Fine-Tuning — descongelar el último bloque

Partimos del modelo TL ya entrenado y liberamos **solo el bloque convolucional final** (`block5_*` de VGG16) para adaptarlo a la textura de las cápsulas, re-entrenando con learning rate muy bajo (`Adam(1e-5)`).

Por qué así: con LR normal, o descongelando toda la base, se destruyen los pesos de ImageNet. Y ojo: después de tocar `trainable` hay que **recompilar** para que tenga efecto.


In [ ]:
base_tl.trainable = True
for layer in base_tl.layers:
    layer.trainable = layer.name.startswith("block5_conv")
print("Capas que ahora aprenden:", [l.name for l in base_tl.layers if l.trainable])

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # LR bajo: ajuste fino, no reescritura
    loss="binary_crossentropy", metrics=["accuracy"],
)

es_ft = EarlyStopping(monitor="val_accuracy", mode="max", patience=3, restore_best_weights=True)

history_ft = model_tl.fit(
    X_train_tl, y_train, epochs=15, batch_size=32,
    validation_split=0.2, callbacks=[es_ft],
)


In [ ]:
acc, val_acc = history_ft.history["accuracy"], history_ft.history["val_accuracy"]
loss, val_loss = history_ft.history["loss"], history_ft.history["val_loss"]
epochs = range(len(acc))

plt.plot(epochs, acc, "bo", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy (FT)")
plt.legend()
plt.figure()
plt.plot(epochs, loss, "bo", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss (FT)")
plt.legend()
plt.show()

print(model_tl.evaluate(X_test_tl, y_test, verbose=0))

y_pred_ft = (model_tl.predict(X_test_tl, verbose=0).ravel() > 0.5).astype(int)
print(classification_report(y_test, y_pred_ft, target_names=["good", "defectuosa"]))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_ft), display_labels=["good", "defectuosa"]
).plot()
plt.title("Matriz de confusión FT (test)")
plt.show()


## 8. Comparativa CNN → TL → FT

Los tres enfoques sobre el **mismo** split de test. Se usan los arrays de predicciones ya guardados (`y_pred`, `y_pred_tl`) más la predicción fresca del modelo fine-tuneado — por eso hay que correr las celdas en orden.

Lo esperable con tan pocos datos: `CNN < TL`, y `FT` igual o un poco mejor que `TL`.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def scores_row(name, yt, yp):
    acc = accuracy_score(yt, yp)
    prec, rec, f1, _ = precision_recall_fscore_support(yt, yp, average="binary", zero_division=0)
    print(f"{name:15s} acc={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")

print("--- test (mismo split) ---")
scores_row("CNN desde cero", y_test, y_pred)
scores_row("TL congelado", y_test, y_pred_tl)
scores_row("FT bloque5", y_test, y_pred_ft)


## 9. Informe (completar según pautas de actividad.md)

- **Problema abordado:** …
- **Dataset empleado:** …
- **Resultados obtenidos:** … (pegar la tabla de §8, curvas y matrices)
- **Conclusiones:** … (¿cuánto ganó TL sobre la CNN? ¿aportó algo FT? ¿qué errores quedan?)
